# Terminal map — global catalog

Starting point for terminal deep-dives. Built from DuckDB trades via:

```bash
python build_terminal_catalog.py
```

**Tables**
- `data/derived/terminals.parquet` — one row per installation
- `data/derived/terminal_products.parquet` — product mix by direction
- `data/derived/terminal_partners.parquet` — top partner countries
- `data/derived/terminals_enrichment_template.csv` — fill later: demand centers, pipelines, lat/lon

**Roles:** `export_only` / `import_only` / `hub` (both).

Later layers: demand centers, regions covered, pipeline connectivity, arb/margins.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
DERIVED = ROOT / "data" / "derived"

terminals = pd.read_parquet(DERIVED / "terminals.parquet")
products = pd.read_parquet(DERIVED / "terminal_products.parquet")
partners = pd.read_parquet(DERIVED / "terminal_partners.parquet")

print(len(terminals), "terminals")
print(terminals["role"].value_counts())
terminals.head()

## Universe snapshot

In [ ]:
by_country = (
    terminals.groupby("country", dropna=False)
    .agg(n_terminals=("terminal_key", "count"), total_mb=("total_mb", "sum"))
    .sort_values("total_mb", ascending=False)
    .head(20)
)
display(by_country)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
terminals["role"].value_counts().plot(kind="bar", ax=axes[0], title="Terminals by role")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
(
    terminals.nlargest(15, "total_mb")
    .set_index("installation_name")[["export_mb", "import_mb"]]
    .plot(kind="barh", stacked=True, ax=axes[1], title="Top 15 terminals (mb)")
)
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## Terminal deep-dive helper

Pick a terminal by name (substring match) or `terminal_key`.

In [ ]:
def find_terminals(query: str, n: int = 15) -> pd.DataFrame:
    q = query.lower()
    hit = terminals[
        terminals["installation_name"].fillna("").str.lower().str.contains(q, regex=False)
        | terminals["country"].fillna("").str.lower().str.contains(q, regex=False)
    ]
    return hit.nlargest(n, "total_mb")[
        [
            "terminal_key",
            "installation_name",
            "country",
            "zone",
            "role",
            "export_mb",
            "import_mb",
            "total_mb",
            "looks_like_refinery",
        ]
    ]


def deep_dive(terminal_key: str):
    meta = terminals.loc[terminals["terminal_key"] == terminal_key]
    if meta.empty:
        print("not found:", terminal_key)
        return
    row = meta.iloc[0]
    print("=" * 72)
    print(row["installation_name"], "|", row["country"], "|", row["role"])
    print(
        f"export {row['export_mb']:.1f} mb · import {row['import_mb']:.1f} mb · "
        f"zone={row['zone']} · trading_region={row['trading_region']}"
    )
    print(
        "Enrichment placeholders —",
        f"demand_centers={row['demand_centers']}, "
        f"regions_covered={row['regions_covered']}, "
        f"pipeline_connected={row['pipeline_connected']}",
    )
    print("=" * 72)

    prod = products[products["terminal_key"] == terminal_key].copy()
    print("\nProducts")
    display(
        prod.pivot_table(
            index=["family", "product"], columns="direction", values="mb", aggfunc="sum"
        )
        .fillna(0)
        .sort_values(by=list(prod["direction"].unique()), ascending=False)
        .head(25)
    )

    part = partners[partners["terminal_key"] == terminal_key].copy()
    print("\nPartner countries")
    display(part.sort_values("mb", ascending=False).head(30))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, direction, title in [
        (axes[0], "export", "Exports by product (mb)"),
        (axes[1], "import", "Imports by product (mb)"),
    ]:
        sub = prod[prod["direction"] == direction].nlargest(10, "mb")
        if sub.empty:
            ax.set_title(title + " — none")
            ax.axis("off")
            continue
        sub.set_index("product")["mb"].plot(kind="barh", ax=ax, title=title)
        ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    return {"meta": row, "products": prod, "partners": part}


# Examples
display(find_terminals("Cartagena"))
display(find_terminals("Sines"))
display(find_terminals("Fos"))

In [ ]:
# Pick a terminal_key from find_terminals() and deep-dive
example = find_terminals("Cartagena", n=1)
if len(example):
    deep_dive(example.iloc[0]["terminal_key"])

## Country slice (e.g. Spain / France / Portugal)

In [ ]:
FOCUS = ["Spain", "France", "Portugal"]
focus = terminals[terminals["country"].isin(FOCUS)].copy()
print(len(focus), "terminals in", FOCUS)
display(
    focus.nlargest(25, "total_mb")[
        [
            "installation_name",
            "country",
            "role",
            "export_mb",
            "import_mb",
            "total_mb",
            "looks_like_refinery",
        ]
    ]
)

## Enrichment workflow (next)

1. Edit `data/derived/terminals_enrichment_template.csv` (or a filtered copy) with:
   - `demand_centers` — cities/industrial basins served
   - `regions_covered` — hinterland / PADD-like region
   - `pipeline_connected` — True/False/Unknown
   - `pipeline_names` — e.g. CLH, SPSE, …
   - `lat` / `lon` — for maps
2. Merge back on `terminal_key`.
3. Later: join arb/margin signals to the same keys.

Kpler alone cannot answer pipeline connectivity or true demand centers — those need
external/reference data layered onto this activity map.